# H4 · Twin loss λ grid + calibration (Himasri)

1. Zero-shot NLI on Rathi's R3 twin files (GPU, ~5 min on a T4).
2. λ grid: λ_inv ∈ {0, 0.5, 1, 2} × λ_sens ∈ {0, 0.5, 1}, seed 42; pick the best by validation twin score with ≤ 1 point macro-F1 loss (CLAUDE.md §3).
3. Final models with seeds 42/43/44: `twin` (chosen λ) and `notwin` (λ = 0) → E2.
4. Temperature scaling + Unverified threshold fitted on validation, applied to test.
5. Push everything to `himasri/H4-calibration`.

**Before you start:** Runtime → **T4 GPU**; Colab secret `GH_TOKEN` (same as for H3). Steps 3–4 run on CPU, about 1–3 min per training run (18 runs).

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name --format=csv
!git clone -q https://github.com/himaallu/Group-Project.git /content/twincheck
%cd /content/twincheck
!git checkout -q himasri/H4-calibration
# Rathi's R2 + R3 files (until they are merged into main); NOT committed from here.
!git fetch -q origin rathi/R3-twins
!git checkout origin/rathi/R3-twins -- data/copies/en/train.jsonl data/copies/en/test.jsonl features/relevance_en.parquet features/entities_en.jsonl features/relevance_copies_en.parquet
!git reset -q data/copies features/relevance_en.parquet features/entities_en.jsonl features/relevance_copies_en.parquet
!pip install -q -r requirements.txt
from google.colab import userdata
!ls -la data/copies/en features features/nli_zeroshot

## 2. 10-row check of the twin NLI scoring
Outputs go to `/content/test10`, not into the repo.

In [ ]:
!python -m src.nli.score_zeroshot data/copies/en/test.jsonl --limit 10 --out-dir /content/test10
!python -c "import pandas as pd; d=pd.read_parquet('/content/test10/test_first10.parquet'); print(d.head()); print(len(d), 'rows')"

## 3. Zero-shot NLI on all twins (~49k pairs, ~5 min)

In [ ]:
!python -m src.nli.score_zeroshot data/copies/en/test.jsonl data/copies/en/train.jsonl --batch-size 64 --out-dir features/nli_zeroshot/copies_en
!ls -la features/nli_zeroshot/copies_en

## 4. λ grid (12 runs, seed 42) → choice → final `twin` / `notwin` with seeds 42/43/44

In [ ]:
!python -u -m src.verdict.grid --no-wandb \
    --twins data/copies/en/train.jsonl data/copies/en/test.jsonl \
    --nli features/nli_zeroshot/multifc_train.parquet features/nli_zeroshot/multifc_val.parquet features/nli_zeroshot/multifc_test.parquet \
          features/nli_zeroshot/copies_en/train.parquet features/nli_zeroshot/copies_en/test.parquet \
    --relevance features/relevance_en.parquet features/relevance_copies_en.parquet 2>&1 | grep --line-buffered -E "^===|chosen|early stop|Error|Traceback"
!cat results/H4/grid_summary.csv
!cat results/H4/selected_lambda.json

## 5. Calibration: fit T and τ on each seed's validation logits, apply to its test predictions

In [ ]:
import shutil
for seed in (42, 43, 44):
    for name in ('twin', 'notwin'):
        print(f'--- {name} seed {seed}')
        !python -m src.verdict.calibrate --val-logits results/H4/val_logits_{name}_s{seed}.jsonl --out models/verdict_calib_{name}_s{seed}.json --apply results/H4/preds_{name}_s{seed}.jsonl results/H4/preds_{name}_cal_s{seed}.jsonl | grep --line-buffered -E '"T"|"tau"|calibrated'
# the pipeline uses the twin model's seed-42 settings
shutil.copy('models/verdict_calib_twin_s42.json', 'models/verdict_calib.json')

## 6. Summary (paste this into the chat with Claude)

In [ ]:
import glob, json, re
import numpy as np, pandas as pd
from src.eval.loaders import load_jsonl, load_preds, join_gold
from src.eval.metrics import compute_all
gold = load_jsonl('data/processed/en/multifc_test.jsonl') + load_jsonl('data/copies/en/test.jsonl')
rows = []
for path in sorted(glob.glob('results/H4/preds_*_s*.jsonl')):
    model, seed = re.match(r'.*preds_(.+)_s(\d+)\.jsonl', path).groups()
    df, _ = join_gold(load_preds(path), gold)
    rows.append({'model': model, 'seed': int(seed), **compute_all(df)})
res = pd.DataFrame(rows)
cols = ['macro_f1', 'auroc', 'ece', 'esr', 'unrelated_rate', 'paraphrase_consistency']
summary = res.groupby('model')[cols].agg(['mean', 'std']).round(4)
pd.set_option('display.width', 250)
print(summary.to_string())
res.to_csv('results/H4/test_metrics_by_seed.csv', index=False)
print(open('results/H4/selected_lambda.json').read())

## 7. Push to `himasri/H4-calibration`
Fill in your GitHub email. Rathi's files and the 12 grid models are not pushed (grid metrics are).

In [ ]:
GIT_NAME = 'Himasri'
GIT_EMAIL = 'YOUR_GITHUB_EMAIL'
token = userdata.get('GH_TOKEN')
!git config user.name "{GIT_NAME}"
!git config user.email "{GIT_EMAIL}"
!git add features/nli_zeroshot/copies_en results/H4 models/verdict_twin_s4* models/verdict_notwin_s4* models/verdict_calib*.json
!git status --short | grep -v '^??'
!git commit -q -m "[H4] Twin NLI features, lambda grid, twin/notwin models (seeds 42/43/44), calibration"
!git push -q https://{token}@github.com/himaallu/Group-Project.git himasri/H4-calibration && echo pushed

When it says `pushed`: Runtime → Disconnect and delete runtime.